In [13]:
# 0. Setup & Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing import load_data, build_preprocessor
from src.mcc import calculate_mcc
from src.zones import apply_mcc_strategy
from src.cost_analysis import run_cost_analysis

# Load data
X, y = load_data("cvd_dataset.csv")
print(f"Dataset loaded: {X.shape[0]} patients, {X.shape[1]} features")

Dataset loaded: 1529 patients, 17 features


In [15]:
# 1. Model Training & Cross-Validation
from src.models import train_and_evaluate, summarise_metrics

preprocessor = build_preprocessor(X)
metrics_df, all_predictions = train_and_evaluate(X, y, preprocessor)

print(summarise_metrics(metrics_df))

Training: Logistic Regression
Training: Decision Tree
Training: Random Forest
Training: Extra Trees
Training: Gradient Boosting
Training: Hist Gradient Boosting
Training: AdaBoost
Training: KNN
Training: Gaussian Naive Bayes
Training: SVC
                        accuracy  balanced_accuracy  f1_macro  auc_macro
model                                                                   
SVC                        0.633              0.603     0.586      0.800
Gaussian Naive Bayes       0.625              0.549     0.547      0.734
Hist Gradient Boosting     0.653              0.537     0.533      0.779
Logistic Regression        0.587              0.541     0.531      0.727
Gradient Boosting          0.664              0.534     0.524      0.775
AdaBoost                   0.664              0.522     0.495      0.742
Random Forest              0.647              0.509     0.489      0.807
Extra Trees                0.617              0.487     0.469      0.754
KNN                        0.60

In [16]:
# 2. MCC Score Calculation
proba_columns   = [c for c in all_predictions["SVC"].columns if c.startswith("proba_")]
predictions_df  = all_predictions["SVC"].copy()

predictions_df["mcc"] = predictions_df.apply(
    lambda row: calculate_mcc(row, proba_columns), axis=1
)

print(f"MCC range: {predictions_df['mcc'].min():.3f} to {predictions_df['mcc'].max():.3f}")
print(f"Mean MCC:  {predictions_df['mcc'].mean():.3f}")

MCC range: 0.016 to 0.998
Mean MCC:  0.711


In [17]:
# 3. Zone Classification (τ = 0.5)
predictions_df_zones = apply_mcc_strategy(predictions_df, mcc_threshold=0.5)

below     = predictions_df_zones[predictions_df_zones["mcc"] < 0.5]
zone_counts = below["zone"].value_counts()

print(f"Patients below threshold (τ=0.5): {len(below)} ({len(below)/len(predictions_df)*100:.1f}%)")
print(f"  Zone 0 - Clinical Review Group  : {zone_counts.get('zone0', 0)}")
print(f"  Zone 1 - No-Action Group        : {zone_counts.get('zone1', 0)}")
print(f"  Zone 2 - Upclassification Group : {zone_counts.get('zone2', 0)}")

Patients below threshold (τ=0.5): 320 (20.9%)
  Zone 0 - Clinical Review Group  : 49
  Zone 1 - No-Action Group        : 59
  Zone 2 - Upclassification Group : 212


In [18]:
# 4. Cost-Effectiveness Analysis
df_standard    = apply_mcc_strategy(predictions_df, mcc_threshold=0.0)
df_uncertainty = apply_mcc_strategy(predictions_df, mcc_threshold=0.5)

fn_standard    = ((df_standard["true"]    == "HIGH") & (df_standard["pred"]    != "HIGH")).sum()
fn_uncertainty = ((df_uncertainty["true"] == "HIGH") & (df_uncertainty["pred"] != "HIGH")).sum()
fp_uncertainty = ((df_uncertainty["pred"] == "HIGH") & (df_uncertainty["true"] != "HIGH")).sum()
zone0_referrals = (df_uncertainty["zone"] == "zone0").sum()
n_total         = len(predictions_df)

bd, de = run_cost_analysis(fn_standard, fn_uncertainty, fp_uncertainty, zone0_referrals, n_total)

print(f"Standard screening   — FN: {fn_standard}")
print(f"Uncertainty-informed — FN: {fn_uncertainty}")
print(f"CVD events prevented : {fn_standard - fn_uncertainty}")
print(f"\nBangladesh total costs: {bd['currency']} {bd['Total costs'][0]:,} → {bd['Total costs'][1]:,}")
print(f"Cost savings: {bd['currency']} {bd['Cost savings']:,} ({bd['Cost savings (%)']:.1f}%)")
print(f"\nGermany total costs: {de['currency']} {de['Total costs'][0]:,} → {de['Total costs'][1]:,}")
print(f"Cost savings: {de['currency']} {de['Cost savings']:,} ({de['Cost savings (%)']:.1f}%)")

Standard screening   — FN: 226
Uncertainty-informed — FN: 152
CVD events prevented : 74

Bangladesh total costs: USD 245,623 → 177,339.6
Cost savings: USD 68,283.4 (27.8%)

Germany total costs: EUR 1,828,547 → 1,278,662
Cost savings: EUR 549,885 (30.1%)
